# Text2SQL — загвартай вэб ба flow builder (Kaggle, GPU T4 x2)

1. Баруун талын **Input**: `text2sql` (код) ба `t2s-adapter-d2` (D2 adapter) dataset нэмэгдсэн байна.
2. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
3. **Run All** (дээд мөр ▶▶). ~30–40 минутын дараа сүүлийн нүдэнд:
   - `FLOW BUILDER: https://….trycloudflare.com/builder?k=…` — алхмуудыг угсарч **▶ Турших**-аар бодит загвараар ажиллуулна
   - `WEB URL: …/?k=…` — энгийн асуулт-хариултын вэб
4. Холбоосыг бүтнээр нь (`?k=` хэсэгтэй) нээнэ. Энэ notebook-ийн таб нээлттэй байх хугацаанд (WEB_MIN, анхдагч 9 цаг) ажиллана.
   Дуусмагц **Stop session** дарж GPU квотоо хэмнэнэ.

In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path
IN = Path("/kaggle/input")
# Kaggle zip-ийг задалж mount хийдэг: код = run_plan_l4.sh агуулсан хавтас; задлаагүй бол t2s_code.zip
code = next((p.parents[2] for p in IN.rglob("scripts/server/run_plan_l4.sh")), None)
REPO = Path("/tmp/t2s")
shutil.rmtree(REPO, ignore_errors=True)
if code is not None:
    shutil.copytree(code, REPO)
else:
    z = next(IN.rglob("t2s_code.zip"), None)
    assert z, "text2sql (код) dataset-ийг Add Input хийнэ үү"
    zipfile.ZipFile(z).extractall(REPO)
assert list(IN.rglob("t2s_adapter_d2.zip.part00")), "t2s-adapter-d2 dataset-ийг Add Input хийнэ үү"
os.chdir(REPO)
(REPO / "config/kaggle_overrides.env").unlink(missing_ok=True)   # өөр stage-ийн тохиргоо — вэбд хэрэггүй
os.environ.update(T2S_PG_PASSWORD="t2s_nb_admin", VLLM_USE_FLASHINFER_SAMPLER="0", ADAPTER_SRC=str(IN),
                  STAGE="web", WEB_MIN=os.environ.get("WEB_MIN", "540"), HF_HUB_ENABLE_HF_TRANSFER="0")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

def run(cmd):
    p = subprocess.Popen(["bash", "-c", f"source scripts/server/env.sh && {cmd}"], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")

In [ ]:
# PostgreSQL 16, Python хамаарал, DB (seed 42), индекс — ~10–15 мин
run("bash scripts/notebook/bootstrap.sh")

In [ ]:
# D2 нэгтгэх + 2 загвар (4-bit, GPU тус бүрт) + вэб + tunnel. «FLOW BUILDER:» ба «WEB URL:» мөр гарна (~15–25 мин).
run("bash scripts/server/run_plan_l4.sh")